In [10]:
import csv
import random
import os

def generate_crop_data(crop_name, n_range, p_range, k_range, temp_range, humidity_range, ph_range, rainfall_range, rows=100):
    """Generate realistic crop data based on agronomic requirements"""
    data = []
    for _ in range(rows):
        n = random.randint(n_range[0], n_range[1])
        p = random.randint(p_range[0], p_range[1])
        k = random.randint(k_range[0], k_range[1])
        temp = round(random.uniform(temp_range[0], temp_range[1]), 8)
        humidity = round(random.uniform(humidity_range[0], humidity_range[1]), 8)
        ph = round(random.uniform(ph_range[0], ph_range[1]), 9)
        rainfall = round(random.uniform(rainfall_range[0], rainfall_range[1]), 7)
        data.append([n, p, k, temp, humidity, ph, rainfall, crop_name])
    return data

# Crop specifications
crops_specs = {
    'turdal':    {'n_range': (20, 40),  'p_range': (50, 70),  'k_range': (30, 50),  'temp_range': (25, 35), 'humidity_range': (60, 75), 'ph_range': (6.0, 7.5), 'rainfall_range': (60, 100)},
    'greengram': {'n_range': (20, 40),  'p_range': (40, 60),  'k_range': (20, 40),  'temp_range': (25, 35), 'humidity_range': (65, 80), 'ph_range': (6.5, 7.5), 'rainfall_range': (60, 100)},
    'tomato':    {'n_range': (80, 120), 'p_range': (60, 100), 'k_range': (100, 140),'temp_range': (18, 27), 'humidity_range': (60, 80), 'ph_range': (6.0, 7.0), 'rainfall_range': (60, 150)},
    'onion':     {'n_range': (80, 120), 'p_range': (50, 80),  'k_range': (50, 80),  'temp_range': (15, 25), 'humidity_range': (60, 70), 'ph_range': (6.0, 7.0), 'rainfall_range': (50, 100)},
    'sugarcane': {'n_range': (100,150), 'p_range': (40, 70),  'k_range': (80, 120), 'temp_range': (25, 35), 'humidity_range': (75, 90), 'ph_range': (6.0, 7.5), 'rainfall_range': (150, 250)},
    'groundnut': {'n_range': (20, 50),  'p_range': (50, 80),  'k_range': (50, 80),  'temp_range': (25, 32), 'humidity_range': (60, 70), 'ph_range': (6.0, 7.0), 'rainfall_range': (50, 125)}
}

# Ensure directory exists
output_dir = os.path.join("notebooks", "data-raw")
os.makedirs(output_dir, exist_ok=True)

final_filename = os.path.join(output_dir, 'synthetic_crop_data.csv')

with open(final_filename, 'w', newline='') as f:
    writer = csv.writer(f)
    # Write header
    writer.writerow(['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label'])
    
    # Generate and write rows for each crop
    for crop_name, specs in crops_specs.items():
        data = generate_crop_data(
            crop_name,
            specs['n_range'],
            specs['p_range'],
            specs['k_range'],
            specs['temp_range'],
            specs['humidity_range'],
            specs['ph_range'],
            specs['rainfall_range']
        )
        writer.writerows(data)

print(f"\n✅ All crop data combined and saved in {final_filename}")



✅ All crop data combined and saved in notebooks\data-raw\synthetic_crop_data.csv


In [ ]:
#label encoder
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
import joblib

# ============================================
# 1️⃣ Load CSV
# ============================================
data_path = os.path.join("C:\\Users\\varsa\\Agropulse\\notebooks\\data\\raw", "climate_crop_data.csv")
df = pd.read_csv(data_path)
print("✅ Dataset loaded successfully!")
print(df.head())

# ============================================
# 2️⃣ Target crops & ideal ranges
# ============================================
target_crops = ['paddy','maize','tur dal','green gram','tomato','onion','sugarcane','banana','coconut','groundnut']

ideal_ranges = {
    'paddy':     {'temperature': (25, 32), 'humidity': (70, 90), 'ph': (6, 7.5), 'rainfall': (150, 250)},
    'maize':     {'temperature': (20, 32), 'humidity': (60, 80), 'ph': (5.5, 7), 'rainfall': (100, 200)},
    'tur dal':   {'temperature': (25, 35), 'humidity': (60, 75), 'ph': (6, 7.5), 'rainfall': (60, 100)},
    'green gram':{'temperature': (25, 35), 'humidity': (65, 80), 'ph': (6.5, 7.5), 'rainfall': (60, 100)},
    'tomato':    {'temperature': (18, 27), 'humidity': (60, 80), 'ph': (6, 7), 'rainfall': (60, 150)},
    'onion':     {'temperature': (15, 25), 'humidity': (60, 70), 'ph': (6, 7), 'rainfall': (50, 100)},
    'sugarcane': {'temperature': (25, 35), 'humidity': (75, 90), 'ph': (6, 7.5), 'rainfall': (150, 250)},
    'banana':    {'temperature': (25, 35), 'humidity': (80, 95), 'ph': (5.5, 7), 'rainfall': (150, 300)},
    'coconut':   {'temperature': (25, 35), 'humidity': (75, 90), 'ph': (5, 7), 'rainfall': (150, 300)},
    'groundnut': {'temperature': (25, 32), 'humidity': (60, 70), 'ph': (6, 7), 'rainfall': (50, 125)}
}

# ============================================
# 3️⃣ Clean labels
# ============================================
df['label'] = df['label'].str.lower().str.strip()
df['label'] = df['label'].replace({'rice':'paddy'})
df = df[df['label'].isin(target_crops)].copy()
print("✅ Filtered dataset:", df['label'].value_counts())

# ============================================
# 4️⃣ Drop N, P, K
# ============================================
df = df.drop(columns=['N','P','K'], errors='ignore')

# ============================================
# 5️⃣ Add season column & encode
# ============================================
crop_season_map = {
    'paddy': 'kharif', 'maize': 'kharif', 'tur dal': 'rabi', 'green gram': 'rabi',
    'tomato': 'summer', 'onion': 'rabi', 'sugarcane': 'year-round', 'banana': 'year-round',
    'coconut': 'year-round', 'groundnut': 'kharif'
}
df['season'] = df['label'].map(crop_season_map)

le_season = LabelEncoder()
df['season'] = le_season.fit_transform(df['season'].astype(str))

# ============================================
# 6️⃣ Compute continuous suitability
# ============================================
def compute_suitability(row):
    crop = row['label']
    ranges = ideal_ranges[crop]
    score = 1
    for feature in ['temperature','humidity','ph','rainfall']:
        low, high = ranges[feature]
        val = row[feature]
        if val < low:
            dist = low - val
        elif val > high:
            dist = val - high
        else:
            dist = 0
        max_range = high - low
        score *= max(0, 1 - dist/max_range)
    return score

df['suitability'] = df.apply(compute_suitability, axis=1)

# ============================================
# 7️⃣ Features & target
# ============================================
X = df[['temperature','humidity','ph','rainfall','season']]
y = df['suitability']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# ============================================
# 8️⃣ Train GBR
# ============================================
gbr = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42)
gbr.fit(X_train, y_train)
print("✅ GBR trained successfully!")

# ============================================
# 9️⃣ Save model & encoders
# ============================================
MODEL_DIR = os.path.join("C:\\Users\\varsa\\Agropulse\\app\\models")
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(gbr, os.path.join(MODEL_DIR, "gbr_crop_suitability.pkl"))
joblib.dump(le_season, os.path.join(MODEL_DIR, "season_encoder.pkl"))
joblib.dump(scaler, os.path.join(MODEL_DIR, "feature_scaler.pkl"))
print(f"✅ Model, encoder & scaler saved at: {MODEL_DIR}")

# ============================================
# 🔟 Predict suitability for all crops
# ============================================
def predict_all_crops(input_features):
    results = {}
    for crop in target_crops:
        temp_features = input_features.copy()
        temp_features['season'] = le_season.transform([crop_season_map[crop]])[0]
        temp_df = pd.DataFrame([temp_features])
        temp_scaled = scaler.transform(temp_df)
        score = gbr.predict(temp_scaled)[0]
        results[crop] = np.clip(score, 0, 1)
    return dict(sorted(results.items(), key=lambda x: x[1], reverse=True))

# Example input
sample_input = {'temperature': 25.5, 'humidity': 78, 'ph': 6.5, 'rainfall': 210}
scores = predict_all_crops(sample_input)

print("\n🌦 Crop Suitability Scores:")
for crop, score in scores.items():
    print(f"  {crop}: {score:.3f}")


✅ Dataset loaded successfully!
    N   P   K  temperature   humidity        ph    rainfall label
0  90  42  43    20.879744  82.002744  6.502985  202.935536  rice
1  85  58  41    21.770462  80.319644  7.038096  226.655537  rice
2  60  55  44    23.004459  82.320763  7.840207  263.964248  rice
3  74  35  40    26.491096  80.158363  6.980401  242.864034  rice
4  78  42  42    20.130175  81.604873  7.628473  262.717340  rice
✅ Filtered dataset: label
paddy        100
maize        100
banana       100
coconut      100
tomato       100
onion        100
sugarcane    100
groundnut    100
Name: count, dtype: int64
✅ GBR trained successfully!
✅ Model, encoder & scaler saved at: C:\Users\varsa\Agropulse\app\models

🌦 Crop Suitability Scores:
  tur dal: 0.992
  green gram: 0.992
  onion: 0.992
  tomato: 0.990
  paddy: 0.976
  maize: 0.976
  groundnut: 0.976
  sugarcane: 0.918
  banana: 0.918
  coconut: 0.918


In [ ]:
#OneHotEncoder-normalize
# ============================================
# 🌾 Crop Suitability Prediction using GBR
# ============================================

import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
import joblib

# ============================================
# 1️⃣ Load CSV
# ============================================
data_path = os.path.join("C:\\Users\\varsa\\Agropulse\\notebooks\\data\\raw", "climate_crop_data.csv")
df = pd.read_csv(data_path)
print("✅ Dataset loaded successfully!")
print(df.head())

# ============================================
# 2️⃣ Target crops & ideal ranges
# ============================================
target_crops = ['paddy','maize','tur dal','green gram','tomato','onion','sugarcane','banana','coconut','groundnut']

ideal_ranges = {
    'paddy':     {'temperature': (25, 32), 'humidity': (70, 90), 'ph': (6, 7.5), 'rainfall': (150, 250)},
    'maize':     {'temperature': (20, 32), 'humidity': (60, 80), 'ph': (5.5, 7), 'rainfall': (100, 200)},
    'tur dal':   {'temperature': (25, 35), 'humidity': (60, 75), 'ph': (6, 7.5), 'rainfall': (60, 100)},
    'green gram':{'temperature': (25, 35), 'humidity': (65, 80), 'ph': (6.5, 7.5), 'rainfall': (60, 100)},
    'tomato':    {'temperature': (18, 27), 'humidity': (60, 80), 'ph': (6, 7), 'rainfall': (60, 150)},
    'onion':     {'temperature': (15, 25), 'humidity': (60, 70), 'ph': (6, 7), 'rainfall': (50, 100)},
    'sugarcane': {'temperature': (25, 35), 'humidity': (75, 90), 'ph': (6, 7.5), 'rainfall': (150, 250)},
    'banana':    {'temperature': (25, 35), 'humidity': (80, 95), 'ph': (5.5, 7), 'rainfall': (150, 300)},
    'coconut':   {'temperature': (25, 35), 'humidity': (75, 90), 'ph': (5, 7), 'rainfall': (150, 300)},
    'groundnut': {'temperature': (25, 32), 'humidity': (60, 70), 'ph': (6, 7), 'rainfall': (50, 125)}
}

# ============================================
# 3️⃣ Clean labels
# ============================================
df['label'] = df['label'].str.lower().str.strip()
df['label'] = df['label'].replace({'rice':'paddy'})
df = df[df['label'].isin(target_crops)].copy()
print("✅ Filtered dataset:", df['label'].value_counts())

# ============================================
# 4️⃣ Drop N, P, K if present
# ============================================
df = df.drop(columns=['N','P','K'], errors='ignore')

# ============================================
# 5️⃣ Add season column & encode
# ============================================
crop_season_map = {
    'paddy': 'kharif', 'maize': 'kharif', 'tur dal': 'rabi', 'green gram': 'rabi',
    'tomato': 'summer', 'onion': 'rabi', 'sugarcane': 'year-round', 'banana': 'year-round',
    'coconut': 'year-round', 'groundnut': 'kharif'
}
df['season'] = df['label'].map(crop_season_map)

le_season = LabelEncoder()
df['season_int'] = le_season.fit_transform(df['season'].astype(str))

# One-hot encode season
ohe_season = OneHotEncoder(sparse_output=False)
season_ohe = ohe_season.fit_transform(df[['season_int']])

# ============================================
# 6️⃣ Compute continuous suitability
# ============================================
def compute_suitability(row):
    crop = row['label']
    ranges = ideal_ranges[crop]
    score = 1
    for feature in ['temperature','humidity','ph','rainfall']:
        low, high = ranges[feature]
        val = row[feature]
        if val < low:
            dist = low - val
        elif val > high:
            dist = val - high
        else:
            dist = 0
        max_range = high - low
        score *= max(0, 1 - dist/max_range)
    return score

df['suitability'] = df.apply(compute_suitability, axis=1)

# ============================================
# 7️⃣ Features & target
# ============================================
X_numeric = df[['temperature','humidity','ph','rainfall']].values
X = np.hstack([X_numeric, season_ohe])
y = df['suitability'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# ============================================
# 8️⃣ Train GBR
# ============================================
gbr = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42)
gbr.fit(X_train, y_train)
print("✅ GBR trained successfully!")

# ============================================
# 9️⃣ Save model & encoders
# ============================================
MODEL_DIR = os.path.join("C:\\Users\\varsa\\Agropulse\\app\\models")
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(gbr, os.path.join(MODEL_DIR, "gbr_crop_suitability.pkl"))
joblib.dump(scaler, os.path.join(MODEL_DIR, "feature_scaler.pkl"))
joblib.dump(ohe_season, os.path.join(MODEL_DIR, "season_ohe.pkl"))
joblib.dump(le_season, os.path.join(MODEL_DIR, "season_encoder.pkl"))
print(f"✅ Model, encoder & scaler saved at: {MODEL_DIR}")

# ============================================
# 🔟 Predict suitability for all crops (normalized)
# ============================================
def predict_all_crops_normalized(input_features):
    results = {}
    for crop in target_crops:
        temp_features = input_features.copy()
        
        # encode season for this crop
        season_int = le_season.transform([crop_season_map[crop]])[0]
        season_ohe_vec = ohe_season.transform([[season_int]])
        
        # combine numeric + season
        temp_array = np.hstack([np.array([[temp_features['temperature'],
                                          temp_features['humidity'],
                                          temp_features['ph'],
                                          temp_features['rainfall']]]), season_ohe_vec])
        
        temp_scaled = scaler.transform(temp_array)
        score = gbr.predict(temp_scaled)[0]
        results[crop] = max(0, score)
    
    # normalize to sum = 1
    total = sum(results.values())
    normalized_scores = {c: (s/total if total > 0 else 0) for c, s in results.items()}
    return dict(sorted(normalized_scores.items(), key=lambda x: x[1], reverse=True))

# Example input
sample_input = {'temperature': 25.5, 'humidity': 78, 'ph': 6.5, 'rainfall': 210}
normalized_scores = predict_all_crops_normalized(sample_input)

print("\n🌦 Crop Suitability Scores (Normalized %):")
for crop, score in normalized_scores.items():
    print(f"  {crop}: {score*100:.1f}%")


✅ Dataset loaded successfully!
    N   P   K  temperature   humidity        ph    rainfall label
0  90  42  43    20.879744  82.002744  6.502985  202.935536  rice
1  85  58  41    21.770462  80.319644  7.038096  226.655537  rice
2  60  55  44    23.004459  82.320763  7.840207  263.964248  rice
3  74  35  40    26.491096  80.158363  6.980401  242.864034  rice
4  78  42  42    20.130175  81.604873  7.628473  262.717340  rice
✅ Filtered dataset: label
paddy        100
maize        100
banana       100
coconut      100
tomato       100
onion        100
sugarcane    100
groundnut    100
Name: count, dtype: int64
✅ GBR trained successfully!
✅ Model, encoder & scaler saved at: C:\Users\varsa\Agropulse\app\models

🌦 Crop Suitability Scores (Normalized %):
  tomato: 11.1%
  tur dal: 10.0%
  green gram: 10.0%
  onion: 10.0%
  paddy: 9.9%
  maize: 9.9%
  groundnut: 9.9%
  sugarcane: 9.7%
  banana: 9.7%
  coconut: 9.7%


c:\Users\varsa\anaconda3\envs\py310\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\varsa\anaconda3\envs\py310\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\varsa\anaconda3\envs\py310\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\varsa\anaconda3\envs\py310\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\varsa\anaconda3\envs\py310\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  